In [ ]:
#Particle lifetime Vs Orbital Spacing 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

rcParams.update({'font.size': 14})

# -------- data --------
f = "/content/drive/MyDrive/Batchrun_all/BatchRun/results.txt"
masses = [100, 200, 300]
MASS, BETA, LOGT = 0, 1, 7

d = np.loadtxt(f, comments="#", delimiter=",", usecols=(MASS, BETA, LOGT))
M, B, L = d[:,0].astype(int), d[:,1], d[:,2]

# -------- figure --------
fig, ax = plt.subplots(3, 1, figsize=(16, 9), sharex=True,
                       gridspec_kw={"hspace": 0.02})

for i, (a, m) in enumerate(zip(ax, masses)):
    bm, lm = B[M==m], L[M==m]
    bu = np.unique(bm)
    mu = np.array([lm[bm==b].mean() for b in bu])
    su = np.array([lm[bm==b].std()  for b in bu])

    a.errorbar(bu, mu, yerr=su, fmt="o", color="red",
               ecolor="red", capsize=3, markersize=4)

    a.text(0.98, 0.80, rf"{m} $M_\oplus$", transform=a.transAxes,
           ha="right", va="top", fontsize=16, fontweight="bold")

    a.grid(True, ls="--", alpha=0.4)
    a.set_ylim(0, 6.5)
    a.set_xlim(2, 30)
    #a.spines["right"].set_visible(False)
    a.spines["top"].set_visible(i == 0)

ax[1].set_ylabel(r"Mean $\log_{10}(t\,[\mathrm{yr}])$",
                 fontsize=18, fontweight="bold")
ax[-1].set_xlabel(r"$\beta$ (mutual Hill radii)",
                  fontsize=18, fontweight="bold")

for a in ax:
    a.tick_params(labelsize=14)

plt.subplots_adjust(left=0.10, right=0.98)
fig.savefig("mean_logt_vs_beta.pdf", bbox_inches="tight",dpi=200)
plt.show()

In [ ]:
#Particle lifetime Vs Orbital Spacing (with fits)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import pandas as pd
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
rcParams.update({'font.size': 14})

# -------- data --------
f = "/content/drive/MyDrive/Batchrun_all/BatchRun/results.txt"
masses = [100, 200, 300]
MASS, BETA, LOGT = 0, 1, 7

d = np.loadtxt(f, comments="#", delimiter=",", usecols=(MASS, BETA, LOGT))
M, B, L = d[:, 0].astype(int), d[:, 1], d[:, 2]

# -------- figure --------
fig, ax = plt.subplots(
    3, 1, figsize=(16, 9), sharex=True,
    gridspec_kw={"hspace": 0.02}
)

# store fit results
linear_rows = []
quadratic_rows = []

for i, (a, m) in enumerate(zip(ax, masses)):
    bm, lm = B[M == m], L[M == m]

    # Unique betas + mean/std
    bu = np.unique(bm)
    mu = np.array([lm[bm == b].mean() for b in bu])
    su = np.array([lm[bm == b].std()  for b in bu])

    # -------- MAIN PLOT --------
    a.errorbar(
        bu, mu, yerr=su,
        fmt="o", color="red",
        ecolor="red", capsize=3, markersize=4, zorder=2
    )

    # -------- LINEAR FIT (3 <= beta <= 5) --------
    mask_low = (bu >= 3) & (bu <= 5)
    if np.sum(mask_low) > 2:
        p_low = np.polyfit(bu[mask_low], mu[mask_low], 1)

        beta_low = np.linspace(3, 5, 200)
        logt_low = p_low[0]*beta_low + p_low[1]

        a.plot(beta_low, logt_low, "k--", lw=4, zorder=5,
               label=r"Linear fit ($3\leq\beta\leq5$)")

        linear_rows.append({
            "Mass (M_earth)": m,
            "Slope (m)": p_low[0],
            "Intercept (b)": p_low[1]
        })

    # -------- QUADRATIC FIT (STABLE SHIFTED FIT) --------
    mask_high = (bu >= 20) & (bu <= 30)

    # OPTIONAL: remove deep resonance dips (uncomment if needed)
    # mask_high = (bu >= 20) & (bu <= 30) & (mu > 3.5)

    if np.sum(mask_high) > 3:
        beta_high_data = bu[mask_high]
        mu_high = mu[mask_high]

        # SHIFT for stability
        x = beta_high_data - 25.0

        # Fit: logt = A(x^2) + B(x) + C
        p_high = np.polyfit(x, mu_high, 2)

        beta_high = np.linspace(20, 30, 300)
        x_plot = beta_high - 25.0
        logt_high = p_high[0]*x_plot**2 + p_high[1]*x_plot + p_high[2]

        a.plot(beta_high, logt_high, "k-", lw=4, zorder=5,
               label=r"Quadratic trend ($20\leq\beta\leq30$)")

        quadratic_rows.append({
            "Mass (M_earth)": m,
            "A (x^2)": p_high[0],
            "B (x)": p_high[1],
            "C (shifted intercept)": p_high[2]
        })

    # -------- PANEL LABEL --------
    a.text(
        0.98, 0.80, rf"{m} $M_\oplus$",
        transform=a.transAxes,
        ha="right", va="top",
        fontsize=16, fontweight="bold"
    )

    # -------- STYLE --------
    a.grid(True, ls="--", alpha=0.4)
    a.set_ylim(0, 6.5)

    # X-axis EXACTLY to 30
    a.set_xlim(2, 30)

    a.spines["right"].set_visible(True)
    a.spines["top"].set_visible(i == 0)

    if i == 0:
        a.legend(loc="lower right", fontsize=12, frameon=False)

    a.minorticks_on()
    a.tick_params(which='major', axis='both',
                  direction='out', length=12.0, width=2.0,
                  labelsize=14)
    a.tick_params(which='minor', axis='both',
                  direction='out', length=6.0, width=1.5)

# -------- SHARED LABELS --------
ax[1].set_ylabel(r"Mean $\log_{10}(t\,[\mathrm{yr}])$",
                 fontsize=18, fontweight="bold")

ax[-1].set_xlabel(r"$\beta$ (mutual Hill radii)",
                  fontsize=18, fontweight="bold")

# clean ticks
ax[-1].set_xticks(np.arange(2, 31, 2))

plt.subplots_adjust(left=0.10, right=0.98)

# -------- SAVE --------
fig.savefig("mean_logt_vs_beta_with_fits_FINAL.pdf",
            bbox_inches="tight", dpi=200)

plt.show()

# -------- TABLE OUTPUT --------
linear_df = pd.DataFrame(linear_rows)
quadratic_df = pd.DataFrame(quadratic_rows)

print("\nLinear fit table: log10(t) = m*beta + b\n")
print(linear_df.to_string(index=False))

print("\nQuadratic fit table (shifted): log10(t) = A(beta-25)^2 + B(beta-25) + C\n")
print(quadratic_df.to_string(index=False))

# Save tables
linear_df.to_csv("linear_fit_table.csv", index=False)
quadratic_df.to_csv("quadratic_fit_table.csv", index=False)

In [ ]:
#Evolution of Semimajor axis plot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

root   = "/content/drive/MyDrive/Final_batch/Batch"
masses = [100, 200, 300]
betas  = [4.0, 7.0, 29.5]

tmin, tmax = 1e2, 1e8
amin, amax = 0.0, 5.0
Ntp_guess  = 10

PS, DPI, fs = 8, 200, 16

def beta_folder(mass, beta):
    return os.path.join(root, f"Mass_pl[{mass}]", f"beta[{beta:.1f}]")

def find_evolution_files(folder):
    files = []
    for name in os.listdir(folder):
        if name.startswith("evolution_tp[") and name.endswith(".txt"):
            files.append(os.path.join(folder, name))
        elif name.startswith("evolution_tp[") and name.endswith("]"):
            files.append(os.path.join(folder, name))
    return sorted(files)

def load_a_track(filepath):
    d = np.genfromtxt(filepath, delimiter=",", comments="#")
    d = np.atleast_2d(d)
    if d.shape[1] < 2:
        return None

    t = d[:, 0]
    a = d[:, 1]

    good = np.isfinite(t) & np.isfinite(a) & (t >= tmin) & (t <= tmax) & (a > 0)
    if not np.any(good):
        return None

    t = t[good]
    a = a[good]
    if len(t) < 2:
        return None
    return t, a

fig, ax = plt.subplots(3, 3, figsize=(21, 13), sharex=True, sharey=True, dpi=DPI)

norm = plt.Normalize(0, 100)
cmap = "viridis"
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

for i, b in enumerate(betas):
    for j, m in enumerate(masses):

        axy = ax[i, j]
        folder = beta_folder(m, b)

        axy.set_xscale("log")
        axy.set_xlim(tmin, tmax)
        axy.set_ylim(amin, amax)

        axy.xaxis.set_major_locator(LogLocator(base=10))
        axy.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2,10)*0.1))
        axy.minorticks_on()

        axy.tick_params(which="major", direction="out", length=10, width=2.5, labelsize=fs)
        axy.tick_params(which="minor", direction="out", length=6,  width=1.8)

        axy.grid(True, which="major", linestyle="--", alpha=0.25)
        axy.grid(True, which="minor", linestyle=":",  alpha=0.15)

        axy.text(0.02, 0.95, rf"$M_p={m}\,M_\oplus,\ \beta={b:.1f}$",
                 transform=axy.transAxes, fontsize=fs, fontweight="bold",
                 ha="left", va="top")

        if j == 0:
            axy.set_ylabel("a [AU]", fontsize=fs, fontweight="bold")
        if i == 2:
            axy.set_xlabel("Time [yr] (log scale)", fontsize=fs, fontweight="bold")

        if not os.path.isdir(folder):
            axy.text(0.5, 0.5, "Missing folder", transform=axy.transAxes,
                     ha="center", va="center", fontsize=fs-2, alpha=0.7)
            continue

        files = find_evolution_files(folder)
        if len(files) == 0:
            axy.text(0.5, 0.5, "No evolution files", transform=axy.transAxes,
                     ha="center", va="center", fontsize=fs-2, alpha=0.7)
            continue

        tracks, t_end = [], []

        for f in files[:Ntp_guess]:
            out = load_a_track(f)
            if out is None:
                continue
            t, a = out
            tracks.append((t, a))
            t_end.append(t[-1])

        if len(t_end) == 0:
            axy.text(0.5, 0.5, "No usable data", transform=axy.transAxes,
                     ha="center", va="center", fontsize=fs-2, alpha=0.7)
            continue

        t_end = np.sort(np.array(t_end))
        Ntot  = len(t_end)

        # board-count survival fraction
        for (t, a) in tracks:
            dead_before_t = np.searchsorted(t_end, t, side="left")
            alive = Ntot - dead_before_t
            surv  = 100.0 * alive / Ntot
            axy.scatter(t, a, s=PS, c=surv, cmap=cmap, norm=norm, lw=0)
            from matplotlib.ticker import LogLocator, NullFormatter

# Force log minor ticks on ALL subplots (sharex can override things)
for axy in ax.ravel():
    axy.set_xscale("log")
    axy.xaxis.set_major_locator(LogLocator(base=10.0, numticks=12))
    axy.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=100))
    axy.xaxis.set_minor_formatter(NullFormatter())  # minor tick labels off (ticks still show)
fig.subplots_adjust(left=0.07, right=0.90, top=0.96, bottom=0.12, wspace=0.10, hspace=0.12)
cb = fig.colorbar(sm, ax=ax, pad=0.02)
cb.set_label("Surviving particles (%)", fontsize=fs, fontweight="bold")
cb.ax.tick_params(which="major", direction="out", length=10, width=2.5, labelsize=fs)
cb.ax.minorticks_on()

#plt.suptitle("Semimajor Axis Evolution colored by Survival Fraction",
  #           fontsize=fs+2, fontweight="bold", y=0.99)

#plt.savefig("semimajoraxis_newdata.pdf", bbox_inches="tight", dpi=DPI)
plt.show()

In [ ]:
#Evolution of eccentricity plot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter, AutoMinorLocator
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# =========================
# PATH + SETTINGS
# =========================
root = "/content/drive/MyDrive/Final_batch/Batch"

masses = [100, 200, 300]
betas  = [4.0, 7.0, 29.5]

tmin, tmax = 1e2, 1e8
emin, emax = 0.0, 1.0
Ntp = 10

PS  = 20
DPI = 200
fs  = 18

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(3, 3, figsize=(21, 13), sharex=True, sharey=True, dpi=DPI)

norm = plt.Normalize(0, 100)
cmap = "viridis"
sm   = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# ---- Explicit log tick locators for x axis
# Major ticks at decades; minor at 2..9 within each decade
x_major = LogLocator(base=10.0, numticks=20)
x_minor = LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=200)

# =========================
# LOOP: beta rows, mass cols
# =========================
for i, b in enumerate(betas):
    for j, m in enumerate(masses):

        axy = ax[i, j]
        folder = os.path.join(root, f"Mass_pl[{m}]", f"beta[{b:.1f}]")

        # ---- axes limits/scales
        axy.set_xscale("log")
        axy.set_xlim(tmin, tmax)
        axy.set_ylim(emin, emax)

        # ---- FORCE log ticks (major + minor) on x-axis
        axy.xaxis.set_major_locator(x_major)
        axy.xaxis.set_minor_locator(x_minor)
        axy.xaxis.set_minor_formatter(NullFormatter())  # keep minor ticks, hide labels

        # ---- y minor ticks (linear)
        axy.yaxis.set_minor_locator(AutoMinorLocator())

        # ---- Make sure minor ticks are actually drawn on x-axis
        # (this is the key part many people miss)
        axy.tick_params(axis="x", which="major", bottom=True, top=False,
                        direction="out", length=12, width=3, labelsize=fs)
        axy.tick_params(axis="x", which="minor", bottom=True, top=False,
                        direction="out", length=6, width=2)

        axy.tick_params(axis="y", which="major",
                        direction="out", length=12, width=3, labelsize=fs)
        axy.tick_params(axis="y", which="minor",
                        direction="out", length=6, width=2)

        axy.text(0.02, 0.95, rf"$M_p={m}\,M_\oplus,\ \beta={b}$",
                 transform=axy.transAxes, fontsize=fs, fontweight="bold",
                 ha="left", va="top")

        if j == 0:
            axy.set_ylabel("e", fontsize=fs, fontweight="bold")
        if i == 2:
            axy.set_xlabel("Time [yr] (log scale)", fontsize=fs, fontweight="bold")

        # ---- folder check
        if not os.path.isdir(folder):
            axy.text(0.5, 0.5, "Missing folder", transform=axy.transAxes,
                     ha="center", va="center", fontsize=fs)
            continue

        # ---- collect evolution files
        files = sorted([f for f in os.listdir(folder)
                        if f.startswith("evolution_tp[") and f.endswith(".txt")])

        if len(files) == 0:
            axy.text(0.5, 0.5, "No evolution files", transform=axy.transAxes,
                     ha="center", va="center", fontsize=fs)
            continue

        # ---- load tracks + end times
        tracks = []
        t_end  = []

        for fname in files[:Ntp]:
            fpath = os.path.join(folder, fname)

            d = np.atleast_2d(np.loadtxt(fpath, comments="#", delimiter=","))
            if d.shape[1] < 3:
                continue

            t = d[:, 0]
            e = d[:, 2]

            good = np.isfinite(t) & np.isfinite(e) & (tmin <= t) & (t <= tmax) & (e >= 0.0)
            if not np.any(good):
                continue

            t = t[good]
            e = e[good]

            tracks.append((t, e))
            t_end.append(t[-1])

        if len(t_end) == 0:
            axy.text(0.5, 0.5, "No usable tracks", transform=axy.transAxes,
                     ha="center", va="center", fontsize=fs)
            continue

        t_end = np.sort(np.array(t_end))
        Ntot  = len(t_end)

        # ---- board-count survival fraction
        for t, e in tracks:
            dead_before_t = np.searchsorted(t_end, t, side="left")
            alive = Ntot - dead_before_t
            surv  = 100.0 * alive / Ntot

            axy.scatter(t, e, s=PS, c=surv, cmap=cmap, norm=norm, lw=0)

# =========================
# COLORBAR + LAYOUT
# =========================
fig.subplots_adjust(left=0.07, right=0.90, top=0.97, bottom=0.12, wspace=0.10, hspace=0.12)
cb = fig.colorbar(sm, ax=ax, pad=0.02)
cb.set_label("Surviving particles (%)", fontsize=fs, fontweight="bold")
cb.ax.tick_params(which="major", direction="out", length=12, width=3, labelsize=fs)
cb.ax.tick_params(which="minor", direction="out", length=6,  width=2)

#plt.suptitle("Eccentricity Evolution colored by Survival Fraction",
 #            fontsize=fs+2, fontweight="bold", y=0.99)

#plt.savefig("eccentricity_evolution.pdf", bbox_inches="tight", dpi=DPI)
plt.show()

In [ ]:
#Particle Lifetime vs Orbital spacing (100Million years)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

rcParams.update({'font.size': 14})

base = "/content/drive/MyDrive/Final_batch"   # <-- your folder
files = {
    100: f"{base}/results_100_long.txt",
    200: f"{base}/results_200_long.txt",
    300: f"{base}/results_300_long.txt",
}

# =========================
# Column indices in your txt
# (matches your screenshot header)
# =========================
BETA_COL = 1
LOGT_COL = 7

# =========================
# Figure
# =========================
fig, ax = plt.subplots(3, 1, figsize=(16, 9), sharex=True,
                       gridspec_kw={"hspace": 0.02})

fs = 14  # tick label size

for i, (a, m) in enumerate(zip(ax, [100, 200, 300])):

    # Load beta and log10_lifetime from that mass file
    d = np.loadtxt(files[m], comments="#", delimiter=",", usecols=(BETA_COL, LOGT_COL))
    bu_raw = d[:, 0]
    lt_raw = d[:, 1]

    # Unique betas + mean/std at each beta
    bu = np.unique(bu_raw)
    mu = np.array([lt_raw[bu_raw == b].mean() for b in bu])
    su = np.array([lt_raw[bu_raw == b].std()  for b in bu])

    # Plot
    a.errorbar(bu, mu, yerr=su, fmt="o", color="red",
               ecolor="red", capsize=3, markersize=4)

    # Mass label
    a.text(0.98, 0.22, rf"{m} $M_\oplus$", transform=a.transAxes,
           ha="right", va="top", fontsize=16, fontweight="bold")

    # Ticks + grid styling (same vibe as your old plot)
    a.minorticks_on()
    a.tick_params(which='major', axis='both',
                  direction='out', length=12.0, width=2.0,
                  labelsize=fs)
    a.tick_params(which='minor', axis='both',
                  direction='out', length=6.0, width=1.5)

    a.grid(True, which='major', ls='--', alpha=0.4)
    a.grid(True, which='minor', ls=':', alpha=0.2)

    a.set_ylim(0, 10)
    a.spines["right"].set_visible(True)
    a.spines["top"].set_visible(i == 0)

# Shared labels
ax[1].set_ylabel(r"Mean $\log_{10}(t\,[\mathrm{yr}])$",
                 fontsize=18, fontweight="bold")
ax[-1].set_xlabel(r"$\beta$ (mutual Hill radii)",
                  fontsize=18, fontweight="bold")

plt.subplots_adjust(left=0.10, right=0.98)

# Save
fig.savefig("mean_logt_vs_beta_1final_batch.pdf", bbox_inches="tight", dpi=200)

plt.show()

In [ ]:
#Mean-Motion Resonance plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# =========================
# FILES
# =========================
base = "/content/drive/MyDrive/Final_batch"
files = {
    100: f"{base}/results_100_long.txt",
    200: f"{base}/results_200_long.txt",
    300: f"{base}/results_300_long.txt",
}

BETA_COL = 1
LOGT_COL = 7

# =========================
# SIMULATION CONSTANTS
# =========================
M_A = 1.08
L_A = 1.519
a_p = np.sqrt(L_A)
M_E = 3.003e-6

# =========================
# RESONANCES
# =========================
res_list = [
    (4, 3),
    (3, 2),
    (5, 3),
    (2, 1),
    (5, 2),
    (3, 1),
]

def beta_for_resonance(mass_Earth, p, q):
    m_p = mass_Earth * M_E
    X = (m_p / (3.0 * M_A))**(1.0/3.0)
    r = p / q
    return (r**(2.0/3.0) - 1.0) / X

def load_mean_std(path):
    d = np.loadtxt(path, comments="#", delimiter=",", usecols=(BETA_COL, LOGT_COL))
    beta_raw = d[:, 0]
    logt_raw = d[:, 1]

    bu = np.unique(beta_raw)
    mu = np.array([logt_raw[beta_raw == b].mean() for b in bu])
    su = np.array([logt_raw[beta_raw == b].std() for b in bu])

    return bu, mu, su

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

beta_min, beta_max = 2.0, 30.0
ymin, ymax = 0, 10

panel_labels = ["(a)", "(b)", "(c)"]
masses = [100, 200, 300]

for i, (ax, mass) in enumerate(zip(axes, masses)):
    bu, mu, su = load_mean_std(files[mass])

    ax.errorbar(
        bu, mu, yerr=su,
        fmt="o", color="red",
        ecolor="red", capsize=3,
        markersize=4
    )

    ax.set_xlim(beta_min, beta_max)
    ax.set_ylim(ymin, ymax)

    # ticks
    ax.minorticks_on()
    ax.tick_params(which="major", direction="out", length=10, width=2, labelsize=13)
    ax.tick_params(which="minor", direction="out", length=5, width=1.5)

    # grid
    ax.grid(True, which="major", ls="--", alpha=0.35)
    ax.grid(True, which="minor", ls=":", alpha=0.2)

    # bold spines
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    # ---- SINGLE separator line between panels ----
    # keep bottom spine for every panel
    ax.spines["bottom"].set_visible(True)

    # keep top spine only for first panel
    if i == 0:
        ax.spines["top"].set_visible(True)
    else:
        ax.spines["top"].set_visible(False)

    # y-label
    ax.set_ylabel(r"Mean $\log_{10}(t\,[\mathrm{yr}])$", fontsize=18, fontweight="bold")

    # panel label
    ax.text(
        0.015, 0.88, panel_labels[i],
        transform=ax.transAxes,
        fontsize=17, fontweight="bold"
    )

    # mass label
    ax.text(
        0.98, 0.08, rf"{mass} $M_\oplus$",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=20, fontweight="bold"
    )

    # resonance guide lines
    for (p, q) in res_list:
        bres = beta_for_resonance(mass, p, q)
        if beta_min <= bres <= beta_max:
            ax.axvline(bres, color="gray", lw=1.2, alpha=0.4)

    # top resonance labels only on top panel
    if i == 0:
        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())

        beta_ticks = []
        tick_labels = []

        for (p, q) in res_list:
            bres = beta_for_resonance(mass, p, q)
            if beta_min <= bres <= beta_max:
                beta_ticks.append(bres)
                tick_labels.append(f"{p}:{q}")

        ax_top.set_xticks(beta_ticks)
        ax_top.set_xticklabels(tick_labels, fontsize=13)
        ax_top.tick_params(direction="out", length=8, width=2)
        ax_top.set_xlabel(
            r"Mean-motion resonances ($P_{\rm tp}:P_p$)",
            fontsize=16, fontweight="bold"
        )

# =========================
# REMOVE TICKS FROM PANEL SEPARATORS
# =========================
for ax in axes[:-1]:
    ax.tick_params(
        axis="x",
        which="both",
        bottom=False, top=False,
        labelbottom=False
    )
    ax.minorticks_off()

# bottom panel keeps x-axis
axes[-1].set_xlabel(r"$\beta$ (Hill-scaled spacing)", fontsize=22, fontweight="bold")
axes[-1].set_xticks([5, 10, 15, 20, 25, 30])

# layout
plt.tight_layout()
plt.subplots_adjust(hspace=0.0)

# save
plt.savefig("combined_resonance_plot_single_separator.pdf", dpi=300, bbox_inches="tight")
plt.savefig("combined_resonance_plot_single_separator.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
#Orbital Snapshot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import rebound
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# =========================
# USER SETTINGS
# =========================
root   = "/content/drive/MyDrive/Final_batch/Batch"
masses = [100, 200, 300]
betas  = [5.0, 15.0, 29.5]
row_names = ["Point A", "Point B", "Point C"]

LIM_ZOOM_AU = 5.0       # inset zoom around Alpha Cen A
PAD_FRAC    = 0.12      # padding around farthest object for wide view

# Where to place in-panel "title" (axes coords)
TITLE_X = 0.03
TITLE_Y = 0.985   # tiny change: a touch lower/safer vs cropping

# Make sure star B never overlaps the in-panel title:
# We'll draw a light white box behind text.
TITLE_BOX = dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.85)

# =========================
# STYLE SETTINGS
# =========================
FIGSIZE = (12.5, 12.5)
DPI = 250

FS_PANEL = 13
FS_AX = 18
FS_TICK = 11

S_STAR_A = 120
S_STAR_B = 190
S_PLANET = 90
S_TP     = 22

COL_STAR_A = "orange"
COL_STAR_B = "0.45"
COL_PLANET = "blue"
COL_TP     = "red"

EDGE_W = 1.2

# =========================
# HELPERS
# =========================
def find_beta_folder(mass_dir, beta):
    if not os.path.isdir(mass_dir):
        return None
    for name in os.listdir(mass_dir):
        if name.startswith("beta[") and name.endswith("]"):
            try:
                val = float(name[5:-1])
                if abs(val - beta) < 1e-9:
                    return os.path.join(mass_dir, name)
            except:
                pass
    return None

def load_final_snapshot(archive_path):
    sa = rebound.Simulationarchive(archive_path)
    sim = sa[-1]
    ps = sim.particles

    starA  = ps[0]
    planet = ps[1]
    starB  = ps[sim.N - 1]

    # Center coordinates on Alpha Cen A
    x0, y0 = starA.x, starA.y

    starA_xy  = (0.0, 0.0)
    planet_xy = (planet.x - x0, planet.y - y0)
    starB_xy  = (starB.x - x0, starB.y - y0)

    xTP, yTP = [], []
    for k in range(2, sim.N - 1):
        p = ps[k]
        xTP.append(p.x - x0)
        yTP.append(p.y - y0)

    return starA_xy, starB_xy, planet_xy, np.array(xTP), np.array(yTP)

def compute_wide_limits(starB_xy, xTP, yTP, planet_xy):
    xs = [0.0, planet_xy[0], starB_xy[0]]
    ys = [0.0, planet_xy[1], starB_xy[1]]

    if xTP.size:
        xs.extend(list(xTP))
        ys.extend(list(yTP))

    max_abs = max(np.max(np.abs(xs)), np.max(np.abs(ys)))
    if max_abs < 1e-9:
        max_abs = 1.0

    lim = (1.0 + PAD_FRAC) * max_abs
    return lim

def draw_scene(ax, starA_xy, starB_xy, planet_xy, xTP, yTP, lim):
    if len(xTP) > 0:
        ax.scatter(xTP, yTP, s=S_TP, c=COL_TP, alpha=0.90, linewidths=0, zorder=2)

    ax.scatter(starA_xy[0], starA_xy[1],
               s=S_STAR_A, c=COL_STAR_A, edgecolor="k", linewidth=EDGE_W, zorder=10)

    ax.scatter(planet_xy[0], planet_xy[1],
               s=S_PLANET, c=COL_PLANET, edgecolor="k", linewidth=EDGE_W, zorder=11)

    ax.scatter(starB_xy[0], starB_xy[1],
               s=S_STAR_B, c=COL_STAR_B, marker="*", edgecolor="k", linewidth=0.9, zorder=9)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(True, alpha=0.18)

def style_ticks_outer_only(ax, show_left, show_bottom):
    ax.minorticks_on()
    ax.tick_params(which="major", direction="out", length=7, width=1.9, labelsize=FS_TICK)
    ax.tick_params(which="minor", direction="out", length=4, width=1.2)

    # Kill top/right ticks everywhere (prevents inner divider tick artifacts)
    ax.tick_params(axis="both", which="both", top=False, right=False, labeltop=False, labelright=False)

    ax.tick_params(axis="y", which="both", left=show_left, labelleft=show_left)
    ax.tick_params(axis="x", which="both", bottom=show_bottom, labelbottom=show_bottom)

def box_every_panel(ax, lw=1.4):
    # Full box around each panel
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(lw)

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(
    3, 3,
    figsize=FIGSIZE,
    dpi=DPI,
    gridspec_kw=dict(wspace=0.0, hspace=0.0)
)

# Global legend (only once)
hA = plt.Line2D([], [], marker='o', linestyle='None',
                markerfacecolor=COL_STAR_A, markeredgecolor='k',
                markersize=np.sqrt(S_STAR_A)/1.6, label="Alpha Cen A")
hB = plt.Line2D([], [], marker='*', linestyle='None',
                markerfacecolor=COL_STAR_B, markeredgecolor='k',
                markersize=np.sqrt(S_STAR_B)/1.6, label="Alpha Cen B")
hP = plt.Line2D([], [], marker='o', linestyle='None',
                markerfacecolor=COL_PLANET, markeredgecolor='k',
                markersize=np.sqrt(S_PLANET)/1.6, label="Planet")
hT = plt.Line2D([], [], marker='o', linestyle='None',
                markerfacecolor=COL_TP, markeredgecolor='none',
                markersize=np.sqrt(S_TP)/1.2, label="Test Particles")

fig.legend(handles=[hA, hB, hP, hT],
           loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.985), fontsize=12,
           handletextpad=0.6, columnspacing=1.2)

fig.supxlabel("x [AU]", fontsize=FS_AX, fontweight="bold", y=0.03)
fig.supylabel("y [AU]", fontsize=FS_AX, fontweight="bold", x=0.03)

# =========================
# PANELS
# =========================
for i, beta in enumerate(betas):
    for j, m in enumerate(masses):
        ax = axes[i, j]

        mass_dir = os.path.join(root, f"Mass_pl[{m}]")
        beta_dir = find_beta_folder(mass_dir, beta)

        if beta_dir is None:
            ax.text(0.5, 0.5, "Missing beta folder",
                    transform=ax.transAxes, ha="center", va="center",
                    color="crimson", fontsize=12)
            box_every_panel(ax)
            style_ticks_outer_only(ax, show_left=(j==0), show_bottom=(i==2))
            continue

        archive_path = os.path.join(beta_dir, "archive.bin")
        if not os.path.exists(archive_path):
            ax.text(0.5, 0.5, "Missing archive.bin",
                    transform=ax.transAxes, ha="center", va="center",
                    color="crimson", fontsize=12)
            box_every_panel(ax)
            style_ticks_outer_only(ax, show_left=(j==0), show_bottom=(i==2))
            continue

        starA_xy, starB_xy, planet_xy, xTP, yTP = load_final_snapshot(archive_path)

        lim = compute_wide_limits(starB_xy, xTP, yTP, planet_xy)

        # main panel
        draw_scene(ax, starA_xy, starB_xy, planet_xy, xTP, yTP, lim)

        # IN-PANEL title (tiny update: 2-line title so it never gets cropped)
        title_line1 = rf"{row_names[i]} | M={m} $M_\oplus$"
        title_line2 = rf"$\beta$ = {beta:.1f}"
        ax.text(
            TITLE_X, TITLE_Y,
            title_line1 + "\n" + title_line2,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=FS_PANEL, fontweight="bold",
            bbox=TITLE_BOX,
            zorder=50
        )

        # box every panel (distinct)
        box_every_panel(ax, lw=1.4)

        # outer ticks only
        style_ticks_outer_only(ax, show_left=(j==0), show_bottom=(i==2))

        # inset zoom (inner region)
        iax = inset_axes(ax, width="36%", height="36%", loc="upper right", borderpad=1.0)
        draw_scene(iax, starA_xy, starB_xy, planet_xy, xTP, yTP, LIM_ZOOM_AU)

        iax.set_xticks([])
        iax.set_yticks([])
        iax.tick_params(left=False, bottom=False, right=False, top=False)

        for sp in iax.spines.values():
            sp.set_linewidth(1.1)
            sp.set_edgecolor("k")

# Layout: tight grid + space for legend and big labels
plt.subplots_adjust(left=0.07, right=0.995, bottom=0.07, top=0.94, wspace=0.0, hspace=0.0)

# IMPORTANT: remove bbox_inches="tight" to prevent title cropping
plt.savefig("orbital_snapshot_BOXED_titles_inside.png", dpi=DPI, pad_inches=0.02)
plt.savefig("orbital_snapshot.pdf", pad_inches=0.02)
plt.show()

print("Saved: orbital_snapshot.pdf")